# Fine-Tuning Qwen 3.5 0.8B with TRL + SarfTok (Google Colab)

This end-to-end Colab notebook packages **SarfTok** as a wheel, installs it, and then fine-tunes **Qwen 3.5 0.8B** with [TRL](https://github.com/huggingface/trl) using ShareGPT-style Arabic data enriched with SarfTok morphology.


## 0. Session Checklist

1. **Runtime → Change runtime type → GPU (A100/L4 preferred).**
2. **Authenticate with Hugging Face (optional but recommended) to lift rate limits.**
3. **If your SarfTok repo is private, create a temporary PAT or mount Google Drive.**


## 1. Install Notebook Dependencies

Colab ships [uv](https://github.com/astral-sh/uv) by default; install all required Python packages up front so later cells can import them without errors.


In [ ]:
!uv pip install -q huggingface-hub trl transformers accelerate bitsandbytes datasets sentencepiece camel-tools


## 2. (Optional) Log in to Hugging Face Hub

If you plan to download gated checkpoints or push adapters back to the Hub, authenticate once per session.


In [ ]:
# (Optional) Log in to Hugging Face Hub to access gated models or push adapters
from huggingface_hub import notebook_login
notebook_login()


## 3. Clone & Install SarfTok

Clone the private SarfTok repository with your `GITHUB_TOKEN`, pull the latest changes if the folder already exists, and install the package with the HF/CAMeL extras.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/melsiddieg/sarftok.git"
WORKDIR = Path("/content")
PKG_DIR = WORKDIR / "sarftok"

def _resolve_token():
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata  # type: ignore

        secret = userdata.get("GITHUB_TOKEN")
        if secret:
            return secret
    except Exception:
        pass
    return None

token = _resolve_token()
if not token:
    raise RuntimeError("GITHUB_TOKEN secret/env var is required to access the private SarfTok repo.")

netrc = Path.home() / ".netrc"
original_netrc = netrc.read_text() if netrc.exists() else None
entry = f"machine github.com\n  login {token}\n  password x-oauth-basic\n"
if original_netrc:
    netrc.write_text(original_netrc.rstrip() + "\n" + entry)
else:
    netrc.write_text(entry)
os.chmod(netrc, 0o600)

def run(cmd):
    subprocess.run(cmd, check=True)

try:
    if PKG_DIR.exists():
        print("Repository already present; pulling latest changes.")
        os.chdir(PKG_DIR)
        run(["git", "fetch", "origin", "main", "--depth=1"])
        run(["git", "reset", "--hard", "origin/main"])
    else:
        os.chdir(WORKDIR)
        run(["git", "clone", "--depth=1", REPO_URL, str(PKG_DIR)])

    os.chdir(PKG_DIR)
    run(["python", "-m", "pip", "install", "-q", ".[camel,hf]"])
finally:
    if original_netrc is None:
        netrc.unlink(missing_ok=True)
    else:
        netrc.write_text(original_netrc)
        os.chmod(netrc, 0o600)
    os.chdir(WORKDIR)



## 4. Configure Models & Tokenizers

We load Qwen 3.5 0.8B in 4-bit and keep its tokenizer separate from SarfTok's morphological pipeline.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset

from sarftok import SarfTokConfig, SarfTokTokenizer
from sarftok.segmenter import ArabicSegmenter
from sarftok.morph_analyzer.camel_wrapper import CamelMorphAnalyzer

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # Update to Qwen/Qwen2.5-0.8B when available
MAX_SEQ_LEN = 1024
bf16 = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=bf16,
    device_map="auto",
    load_in_4bit=True,
)
qwen_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

sarftok_cfg = SarfTokConfig(
    analyzer_backend="heuristic",
    norm_mode="classical_soft",
    surface_vocab_size=32000,
)
sarftok_tokenizer = SarfTokTokenizer(sarftok_cfg)
sarftok_segmenter = ArabicSegmenter()
sarftok_analyzer = CamelMorphAnalyzer(db_name="calima-msa-r13", top_k=2)


## 5. Build the SarfTok-Enriched Dataset

We pull 3,000 ShareGPT-Arabic chats, summarize their morphology via SarfTok, and store the enriched text in a column named `text` for TRL's `SFTTrainer`.


In [ ]:
DATASET_ID = "FreedomIntelligence/sharegpt-arabic"
MAX_SAMPLES = 3000

raw_dataset = load_dataset(DATASET_ID, split=f"train[:{MAX_SAMPLES}]")


def describe_morph(text: str) -> str:
    words = sarftok_segmenter.segment_flat(text)
    analyses = sarftok_analyzer.analyze_sentence(words, context=text)
    summaries = []
    for word, hyp in zip(words, analyses):
        if not hyp:
            continue
        best = hyp[0]
        parts = []
        if best.root:
            parts.append(f"جذر={best.root}")
        if best.pattern:
            parts.append(f"وزن={best.pattern}")
        if best.pos:
            parts.append(f"نوع={best.pos}")
        if parts:
            summaries.append(f"{word}: " + ", ".join(parts))
    if not summaries:
        return text
    return text + "

### التحليل الصرفي (SarfTok)
" + "
".join(summaries[:32])


def format_dialog(example):
    conv = example.get("conversations") or example.get("messages") or []
    turns = []
    for turn in conv:
        speaker = (turn.get("from") or turn.get("role") or "user").lower()
        tag = "### السؤال" if speaker in {"user", "human"} else "### الإجابة"
        turns.append(f"{tag}
{turn['value'].strip()}")
    text = "

".join(turns)
    return describe_morph(text)

processed = raw_dataset.map(lambda ex: {"text": format_dialog(ex)})
print("Processed records:", len(processed))


## 6. Configure QLoRA (PEFT) + TrainingArguments

We keep rank small to stay within Colab memory and train for 3 epochs over the processed dataset.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="qwen35_sarftok_trl",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=torch.cuda.is_available(),
    num_train_epochs=3,
    logging_steps=10,
    max_steps=-1,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=qwen_tokenizer,
    train_dataset=processed,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    peft_config=lora_config,
    args=training_args,
)
trainer.train()
trainer.save_model("qwen35_sarftok_trl")


## 7. Test Generation

Load the adapter-weighted model and ask a short medical prompt to verify behavior.


In [ ]:
from transformers import pipeline

peft_model = AutoModelForCausalLM.from_pretrained("qwen35_sarftok_trl", device_map="auto")
peft_tokenizer = qwen_tokenizer

chat = pipeline("text-generation", model=peft_model, tokenizer=peft_tokenizer, max_length=256)
prompt = "### السؤال
اشرح فوائد التعلم العميق في تشخيص الأمراض المزمنة.

### الإجابة
"
print(chat(prompt)[0]['generated_text'])


## 8. (Alternative) True Embedding Fusion with `HybridSarfTokCausalLM`

The section above injects morphology as a **text hint** appended to each training example.
SarfTok also supports **embedding-level fusion**: the per-word morphology vector is added
directly onto the base model's token embeddings, leaving every transformer weight untouched.

`SarfTokBaseAdapter` aligns SarfTok morphology to **Qwen's own tokenizer** (via
`word_ids()`), so an existing checkpoint can be morph-fused without retraining a surface
vocabulary. The cell below runs one fused forward/backward step; for a full run, wrap
`fusion_model` in a `Trainer`/`SFTTrainer` configured with `remove_unused_columns=False`
and this `collator`. See `examples/finetune_existing_llm.py` for a standalone version.

In [ ]:
from sarftok.config import SarfTokConfig
from sarftok.llm_integration.base_tokenizer_adapter import SarfTokBaseAdapter
from sarftok.llm_integration.hf_data_collator import SarfTokDataCollator
from sarftok.llm_integration.hf_modeling_embeddings import HybridSarfTokCausalLM
from sarftok.morph_vocab import MorphVocab

# Embedding-fusion config (orthogonality loss on, entropy gating on).
fusion_cfg = SarfTokConfig(
    analyzer_backend="heuristic",   # or "camel" for the richer analyzer
    norm_mode="classical_soft",
    top_k=3,
    alpha=0.5,
    entropy_gating=True,
    ortho_lambda=0.01,
)
morph_vocab = MorphVocab(root_min_freq=1, pattern_min_freq=1)

# Align morphology to Qwen's OWN tokenizer, then fuse onto its embeddings.
adapter = SarfTokBaseAdapter(qwen_tokenizer, fusion_cfg)
collator = SarfTokDataCollator(pad_token_id=qwen_tokenizer.pad_token_id or 0)
fusion_model = HybridSarfTokCausalLM(model, fusion_cfg, morph_vocab)

# Build one small batch from the raw ShareGPT-Arabic conversations.
sample_texts = []
for ex in raw_dataset.select(range(8)):
    conv = ex.get("conversations") or ex.get("messages") or []
    if conv:
        sample_texts.append(conv[0]["value"])

features = [adapter.encode_sentence(t, max_length=MAX_SEQ_LEN) for t in sample_texts]
batch = collator(features)

out = fusion_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
    word_to_surface_spans=batch["word_to_surface_spans"],
    morph_analyses=batch["morph_analyses"],
)
loss = out["loss"] if isinstance(out, dict) else out.loss
loss.backward()
print("fused LM loss:", float(loss))
if isinstance(out, dict) and out.get("ortho_loss") is not None:
    print("orthogonality loss:", float(out["ortho_loss"]))
print("morph gradient present:",
      fusion_model.hybrid_embedding.morph_encoder.rootchar_emb.weight.grad is not None)